# Genetic Wake-Word Search
> **One notebook, any wake word, any hardware.**  
> Synthesise a dataset, search for the best hyperparameters, train multiple model sizes, export to ONNX, and benchmark — all from a single string.

---

## What this notebook does

| Step | Cell | What happens |
|------|------|--------------|
| 1. Install | 3 | Installs all Python dependencies; auto-detects platform (Kaggle / Paperspace / Colab / local) |
| 2. Dataset | 4 | Generates or loads a labelled audio dataset (positives + negatives) |
| 3. Search | 5 | Runs a genetic algorithm to find the best learning rate, batch size, and model width |
| 4. Visualise | 6 | Plots F1 score evolution across generations |
| 5. Train | 7 | Trains one final model per requested tier using the best hyperparameters |
| 6. Compare | 8 | Bar chart of F1 across all trained tiers |
| 7. Benchmark | 9 | Measures latency and parameter count for every model architecture |
| 8. Summary | 10 | Markdown table of results + ONNX export status |

---

## Quick start

**Minimum viable run (no configuration needed):**
1. Open this notebook
2. Click **Run All**
3. Wait ~10 minutes — outputs land in `./ww_output/`

**Change the wake word:**
```
# In a terminal / Kaggle secret / env var:
export WAKE_WORD="hey nova"
# Then Run All
```

---

## Configuration reference

**Only the Config cell (Cell 2) ever needs editing.** Every knob is also controllable via an environment variable — set them as Kaggle Secrets, Paperspace env vars, or shell exports before running.

### Core
| Variable | Default | Description |
|----------|---------|-------------|
| `WAKE_WORD` | `hey jarvis` | The phrase to detect. Any language, any number of words. |
| `OUTPUT_DIR` | `./ww_output` | All outputs (datasets, models, plots) land here. |
| `DEVICE` | `auto` | `auto` picks CUDA > MPS > CPU. Force with `cuda` or `cpu`. |
| `SEED` | `42` | Random seed — set the same value to get reproducible results. |

### Dataset generation
| Variable | Default | Description |
|----------|---------|-------------|
| `LANG_CODE` | `en` | BCP-47 language for TTS synthesis (e.g. `pt`, `de`, `fr`). |
| `N_POSITIVE` | `200` | Number of positive (wake-word) audio clips to synthesise. More = better model, slower datagen. |
| `ADVERSARIAL` | `true` | Adds grapheme-level confusable phrases as hard negatives (e.g. 'hey harris'). Strongly recommended. |
| `DOWNLOAD_AUGMENT` | `false` | Downloads bg-noise, music, and RIR datasets from HuggingFace for augmentation. Slow on first run; cached after. |

### Dataset overrides — bring your own data
| Variable | Format | When to use |
|----------|--------|-------------|
| `CUSTOM_TRAIN_CSV` | `/abs/path/metadata.csv` | You already have labelled audio. Format: `path,label` (no header, label = 1/0). |
| `CUSTOM_TEST_CSV` | `/abs/path/test.csv` | Optional. If absent, the notebook auto-splits your train CSV 80/20. |
| `HF_DATASET` | `org/repo` | Force a specific HuggingFace dataset as the source of positives. Overrides auto-detection. |
| `NEGATIVES_DIR` | `/abs/path/` | Local audio directory used as general negatives — skips HF download. |
| `EXTRA_NEGATIVES_HF` | `org/r1,org/r2` | Append extra HF repos to the built-in general negatives. |
| `BG_NOISE_DIR` | `/abs/path/` | Local directory for bg-noise augmentation — skips HF download. |
| `EXTRA_BG_NOISE_HF` | `org/r1,org/r2` | Append extra HF repos to the built-in bg-noise list. |
| `MUSIC_DIR` | `/abs/path/` | Local directory for music augmentation. |
| `EXTRA_MUSIC_HF` | `org/r1,org/r2` | Append extra HF repos to the built-in music list. |
| `RIR_DIR` | `/abs/path/` | Local directory for room-impulse-response augmentation. |
| `EXTRA_RIR_HF` | `org/r1,org/r2` | Append extra HF repos to the built-in RIR list. |

### Genetic search
| Variable | Default | Description |
|----------|---------|-------------|
| `POPULATION` | `12` | Candidate hyperparameter sets evaluated per generation. Higher = more thorough, slower. |
| `GENERATIONS` | `5` | Number of evolutionary generations. Each generation selects the best half and mutates. |
| `EPOCHS_PER_TRIAL` | `3` | Training epochs for each candidate during search. Keep low for speed; accuracy matters less here than ranking. |
| `SEARCH_FULL` | `false` | `true` enables the full search space (more hyperparameters). Significantly slower. |

### Final training
| Variable | Default | Description |
|----------|---------|-------------|
| `TIERS_TO_TRAIN` | `micro,small,filterbank_small` | Comma-separated list of architecture tiers. See the tier table below. |
| `FINAL_EPOCHS` | `30` | Training epochs for each final model. More = better convergence. |
| `EXPORT_ONNX` | `true` | Export the best checkpoint to ONNX after training. Required for deployment. |

---

## Model tiers

Tiers control the architecture (classifier) and feature extractor. Pick based on your target hardware.

| Tier name | Extractor | Target hardware |
|-----------|-----------|----------------|
| `micro` | MFCC | Microcontroller / RPi Zero |
| `small` | MFCC | RPi 3/4, low-power SBC |
| `medium` | MFCC | Desktop CPU |
| `large` | MFCC | GPU server |
| `filterbank_small` | Filterbank | Embedded, slightly richer features than MFCC |
| `filterbank_medium` | Filterbank | Mid-range CPU |
| `melspec_small` | Mel-spectrogram | CPU with more RAM |
| `melspec_medium` | Mel-spectrogram | GPU recommended |
| `hubert_small` | HuBERT (frozen) | GPU required |
| `hubert_medium` | HuBERT (frozen) | GPU required, best accuracy |
| `whisper_small` | Whisper encoder | GPU required |
| `markov` | Markov chain | Extremely lightweight, CPU-only, no training |

Set `TIERS_TO_TRAIN=micro,small,filterbank_small` for a quick multi-tier comparison.

---

## Resume safety

The notebook is **safe to re-run at any point** after a crash or kernel restart:

- **BYO CSV mode**: the 80/20 split is written once; subsequent runs reuse it.
- **HF / Auto mode**: `reuse_dataset=True` is always set — if the dataset already exists it is loaded, not re-synthesised.
- **Training**: each tier writes to its own subdirectory. Re-running a completed tier re-uses existing weights (no wasted compute).

---

## Outputs

After a full run, `OUTPUT_DIR` contains:

```
ww_output/
├── dataset/                  # generated audio + metadata CSVs
│   ├── train/metadata.csv
│   └── test/metadata.csv
├── genetic/                  # search checkpoints per generation
├── model_micro/              # final micro tier
│   ├── best_f1.pt            # PyTorch checkpoint
│   └── best_f1.onnx          # ONNX model (if EXPORT_ONNX=true)
├── model_small/              # ... one dir per TIERS_TO_TRAIN entry
├── benchmark/                # latency + param count CSVs and PNGs
├── evolution.png             # F1 over generations
└── tier_comparison.png       # F1 bar chart across tiers
```

---

## Platform notes

| Platform | Recommended settings |
|----------|---------------------|
| **Kaggle** (GPU T4) | `DEVICE=cuda`, `N_POSITIVE=500`, `DOWNLOAD_AUGMENT=true`, set secrets in Add-ons → Secrets |
| **Paperspace** (GPU) | same as Kaggle; `OUTPUT_DIR=/notebooks/ww_output` |
| **Google Colab** | `DEVICE=cuda`, mount Drive for persistence (`OUTPUT_DIR=/content/drive/MyDrive/ww_output`) |
| **Local CPU** | defaults work; reduce `N_POSITIVE=50`, `POPULATION=4`, `GENERATIONS=2` for a quick test |
| **Local GPU** | `DEVICE=cuda`; full defaults are fine |

## Cell 2 — Configuration
This is the **only cell you need to edit**. All values can also be set as environment variables.

In [ ]:
import os

WAKE_WORD          = os.environ.get("WAKE_WORD",          "hey jarvis")
OUTPUT_DIR         = os.environ.get("OUTPUT_DIR",         "./ww_output")
LANG               = os.environ.get("LANG_CODE",          "en")
N_POSITIVE         = int(os.environ.get("N_POSITIVE",     "200"))
ADVERSARIAL        = os.environ.get("ADVERSARIAL",        "true").lower()  == "true"
DOWNLOAD_AUGMENT   = os.environ.get("DOWNLOAD_AUGMENT",   "false").lower() == "true"
# Dataset overrides (leave empty to use auto mode)
CUSTOM_TRAIN_CSV   = os.environ.get("CUSTOM_TRAIN_CSV",   "")  # path,label CSV
CUSTOM_TEST_CSV    = os.environ.get("CUSTOM_TEST_CSV",    "")  # optional; 80/20 split if absent
HF_DATASET         = os.environ.get("HF_DATASET",         "")  # e.g. "org/repo"
# Negative / augmentation overrides (all optional, all stackable)
NEGATIVES_DIR      = os.environ.get("NEGATIVES_DIR",      "")  # local audio dir
EXTRA_NEGATIVES_HF = os.environ.get("EXTRA_NEGATIVES_HF", "")  # "org/r1,org/r2"
BG_NOISE_DIR       = os.environ.get("BG_NOISE_DIR",       "")
EXTRA_BG_NOISE_HF  = os.environ.get("EXTRA_BG_NOISE_HF",  "")
MUSIC_DIR          = os.environ.get("MUSIC_DIR",          "")
EXTRA_MUSIC_HF     = os.environ.get("EXTRA_MUSIC_HF",     "")
RIR_DIR            = os.environ.get("RIR_DIR",            "")
EXTRA_RIR_HF       = os.environ.get("EXTRA_RIR_HF",       "")
# Genetic search knobs
POPULATION         = int(os.environ.get("POPULATION",     "12"))
GENERATIONS        = int(os.environ.get("GENERATIONS",    "5"))
EPOCHS_PER_TRIAL   = int(os.environ.get("EPOCHS_PER_TRIAL", "3"))
SEARCH_FULL        = os.environ.get("SEARCH_FULL",        "false").lower() == "true"
# Final model training
TIERS_TO_TRAIN     = os.environ.get("TIERS_TO_TRAIN",     "micro,small,filterbank_small").split(",")
FINAL_EPOCHS       = int(os.environ.get("FINAL_EPOCHS",   "30"))
EXPORT_ONNX        = os.environ.get("EXPORT_ONNX",        "true").lower()  == "true"
DEVICE             = os.environ.get("DEVICE",             "auto")
SEED               = int(os.environ.get("SEED",           "42"))

## Cell 3 — Install & platform detection
Installs all required packages and auto-detects the current platform.
**Safe to skip** if running in an environment where `ww_trainer` and the OVOS stack are already installed.

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "librosa", "onnx", "onnxruntime", "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "ovos-vad-plugin-silero", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

import os
_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)
print(f"Platform: {_platform} | Device: {DEVICE} | Wake word: {WAKE_WORD!r}")

## Cell 4 — Dataset

Loads or generates the labelled audio dataset. Three modes are tried in order:

1. **BYO CSV** (`CUSTOM_TRAIN_CSV` set) — uses your existing `path,label` CSV directly.  
   If `CUSTOM_TEST_CSV` is not set, an 80/20 split is written once to `OUTPUT_DIR/dataset_split/` and reused on subsequent runs.

2. **HF override** (`HF_DATASET` set) — downloads positives from the specified HuggingFace repo instead of the auto-detected one.

3. **Auto** (default) — for known wake words (e.g. *hey jarvis*, *hey mycroft*) positives are downloaded from HF; for unknown wake words they are synthesised via TTS.  
   Either way, `reuse_dataset=True` means re-running this cell never re-synthesises audio that already exists.

**Augmentation overrides** (`BG_NOISE_DIR`, `MUSIC_DIR`, `RIR_DIR`) replace the corresponding HF downloads with local audio.  
**Extra HF repos** (`EXTRA_*_HF`) are appended to the built-in lists — both can be set simultaneously.

Dataset logic lives in `notebooks/nb_dataset.py` — import it to reuse these helpers outside the notebook.

In [ ]:
import sys
from pathlib import Path

# Make nb_dataset importable when running from the notebooks/ directory
sys.path.insert(0, str(Path("__file__").parent if "__file__" in dir() else Path(".")))
from nb_dataset import load_byo, load_generated, apply_local_overrides

if CUSTOM_TRAIN_CSV:
    datagen_result = load_byo(
        train_csv=CUSTOM_TRAIN_CSV, test_csv=CUSTOM_TEST_CSV,
        output_dir=OUTPUT_DIR, seed=SEED,
        negatives_dir=NEGATIVES_DIR,
        bg_noise_dir=BG_NOISE_DIR, music_dir=MUSIC_DIR, rir_dir=RIR_DIR,
    )
    print("Mode: BYO CSV")
else:
    datagen_result = load_generated(
        wake_word=WAKE_WORD, output_dir=OUTPUT_DIR,
        n_positive=N_POSITIVE, lang=LANG,
        adversarial=ADVERSARIAL, download_augmentation=DOWNLOAD_AUGMENT,
        seed=SEED, hf_dataset=HF_DATASET,
        extra_negatives_hf=EXTRA_NEGATIVES_HF,
        extra_bg_noise_hf=EXTRA_BG_NOISE_HF,
        extra_music_hf=EXTRA_MUSIC_HF,
        extra_rir_hf=EXTRA_RIR_HF,
    )
    apply_local_overrides(
        datagen_result,
        negatives_dir=NEGATIVES_DIR,
        bg_noise_dir=BG_NOISE_DIR, music_dir=MUSIC_DIR, rir_dir=RIR_DIR,
        output_dir=OUTPUT_DIR,
        extra_bg_noise_hf=EXTRA_BG_NOISE_HF,
        extra_music_hf=EXTRA_MUSIC_HF,
        extra_rir_hf=EXTRA_RIR_HF,
        download_augmentation=DOWNLOAD_AUGMENT,
    )
    print(f"Mode: {'HF override' if HF_DATASET else 'Auto'}")

n_train = sum(1 for _ in open(datagen_result.train_csv))
n_test  = sum(1 for _ in open(datagen_result.test_csv))
print(f"Train: {n_train} samples | Test: {n_test} samples")

## Cell 5 — Genetic hyperparameter search

Searches for the best combination of learning rate, batch size, and model width using a genetic algorithm:

1. Initialise a **population** of random hyperparameter sets.
2. Train each candidate for `EPOCHS_PER_TRIAL` epochs on 80% of the train CSV.
3. Evaluate on the remaining 20% (the test CSV is never touched during search).
4. Keep the top 50%, mutate survivors, fill the rest with new random candidates.
5. Repeat for `GENERATIONS` generations.

The best `lr` and `batch_size` are forwarded to final training (Cell 7).  
Increase `POPULATION` and `GENERATIONS` for a more thorough search at the cost of time.  
Set `SEARCH_FULL=true` to expand the search space to include additional regularisation and architecture knobs.

In [ ]:
from ww_trainer.sweep import run_genetic_search

genetic_result = run_genetic_search(
    metadata_csv=str(datagen_result.train_csv),
    population_size=POPULATION,
    generations=GENERATIONS,
    epochs_per_trial=EPOCHS_PER_TRIAL,
    featurizer_type="mfcc",
    device=DEVICE,
    output_dir=str(Path(OUTPUT_DIR) / "genetic"),
    full=SEARCH_FULL,
)
best_hp  = genetic_result["best_config"]
best_f1  = genetic_result["best_score"]
print(f"Best search F1 : {best_f1:.4f}")
print(f"Best config    : {best_hp}")

## Cell 6 — Evolution history

Line chart of best and average F1 per generation.  
A steadily rising best-F1 curve means the search is making progress; a flat curve suggests increasing `POPULATION` or `SEARCH_FULL=true` may help.  
Plot saved to `OUTPUT_DIR/evolution.png`.

In [ ]:
import matplotlib.pyplot as plt

history = genetic_result["history"]
gens  = [h["generation"] for h in history]
bests = [h["best"]       for h in history]
avgs  = [h["avg"]        for h in history]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gens, bests, "o-",  label="Best F1")
ax.plot(gens, avgs,  "s--", label="Avg F1",  alpha=0.7)
ax.set_xlabel("Generation"); ax.set_ylabel("F1 (search)")
ax.set_title(f"Genetic Search Evolution \u2014 {WAKE_WORD!r}")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "evolution.png", dpi=120)
plt.show()

## Cell 7 — Final model training

Trains one model per tier in `TIERS_TO_TRAIN` using:
- The **best `lr` and `batch_size`** found by the genetic search.
- The **full train split** (no held-out portion — search is done).
- Evaluation on the untouched **test CSV**.

Each tier produces:
- `best_f1.pt` — PyTorch checkpoint with the highest validation F1.
- `best_f1.onnx` — ONNX export (if `EXPORT_ONNX=true`).

The augmentation dirs found during datagen (`bg_noise_dir`, `music_dir`, `rir_dir`) are  
automatically forwarded to the trainer for online augmentation during training.

In [ ]:
from ww_trainer.quickstart import QuickstartConfig, _train_from_datagen_result

final_results = []
for tier_name in TIERS_TO_TRAIN:
    tier_cfg = QuickstartConfig(
        wake_word=WAKE_WORD,
        output_dir=Path(OUTPUT_DIR) / f"model_{tier_name}",
        tier=tier_name,
        epochs=FINAL_EPOCHS,
        batch_size=best_hp.get("batch_size", 16),
        lr=best_hp.get("lr", 5e-4),
        export_onnx=EXPORT_ONNX,
        device=DEVICE,
        download_augmentation=False,  # already done in Cell 4
        seed=SEED,
    )
    result = _train_from_datagen_result(tier_cfg, datagen_result)
    final_results.append({
        "tier": tier_name,
        "f1":   result.metrics.get("f1", 0.0),
        "onnx": result.best_onnx_path,
        "pt":   result.best_model_path,
    })
    print(f"  {tier_name:20s}  F1={result.metrics.get('f1', 0):.3f}")

## Cell 8 — Tier comparison

Bar chart of test-set F1 for each trained tier.  
Use this to pick the right accuracy/size trade-off for your deployment target.  
Saved to `OUTPUT_DIR/tier_comparison.png`.

In [ ]:
names  = [r["tier"] for r in final_results]
scores = [r["f1"]   for r in final_results]

fig, ax = plt.subplots(figsize=(max(5, len(names) * 1.8), 4))
bars = ax.bar(names, scores, color=plt.cm.tab10.colors[:len(names)])
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=10)
ax.set_ylim(0, 1.1); ax.set_ylabel("F1")
ax.set_title(f"Tier Comparison \u2014 {WAKE_WORD!r}")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "tier_comparison.png", dpi=120)
plt.show()

## Cell 9 — Architecture benchmark

Measures **inference latency** and **parameter count** for every available model architecture  
(not just the tiers you trained — this covers the full tier catalogue so you can compare).  

Outputs CSVs and multiple PNGs to `OUTPUT_DIR/benchmark/`.  
All PNGs are displayed inline below.

In [ ]:
from ww_trainer.benchmark import run_benchmark, plot_results, save_results
from IPython.display import Image, display

bench_dir = Path(OUTPUT_DIR) / "benchmark"
bench_dir.mkdir(parents=True, exist_ok=True)
bench_report = run_benchmark(device=DEVICE, output_dir=str(bench_dir))
save_results(bench_report, bench_dir)
plot_results(bench_report, bench_dir)

for png in sorted(bench_dir.glob("*.png")):
    display(Image(str(png)))

## Cell 10 — Results summary

Markdown table of F1 scores and ONNX export status for every trained tier.  
Copy the ONNX path from the table into your deployment pipeline.

In [ ]:
from IPython.display import Markdown, display

rows = ["| Tier | F1 | ONNX |", "|------|-----|------|"] + [
    f"| {r['tier']} | {r['f1']:.3f} | {'\u2713' if r['onnx'] and Path(r['onnx']).exists() else '\u2014'} |"
    for r in final_results
]
display(Markdown("\n".join(rows)))
print(f"\nAll outputs saved to: {Path(OUTPUT_DIR).resolve()}")

## Next Steps

### Improve accuracy
- Increase `N_POSITIVE` (500+ for production models)
- Set `DOWNLOAD_AUGMENT=true` to add background noise, music, and room reverb during training
- Set `ADVERSARIAL=true` (default) — adds hard confusable negatives
- Set `SEARCH_FULL=true` and increase `POPULATION` / `GENERATIONS` for a deeper search

### Bring your own data
- `CUSTOM_TRAIN_CSV=/path/to/metadata.csv` — skip TTS entirely
- `NEGATIVES_DIR=/path/to/neg/` — custom not-wake-word audio
- `BG_NOISE_DIR`, `MUSIC_DIR`, `RIR_DIR` — local augmentation audio
- `EXTRA_NEGATIVES_HF=org/repo1,org/repo2` — additional HF negative sources

### Change the model family
- `TIERS_TO_TRAIN=micro,small,medium,large` — MFCC classifiers, all sizes
- `TIERS_TO_TRAIN=hubert_small,hubert_medium` — HuBERT-based (GPU required)
- `TIERS_TO_TRAIN=whisper_small` — Whisper encoder backbone (GPU required)
- `TIERS_TO_TRAIN=markov` — zero training, Markov chain model, ultra-lightweight

### Deploy
- ONNX models in `OUTPUT_DIR/model_<tier>/best_f1.onnx` drop into any OVOS plugin
- See [deployment guide](../docs/training.md) for plugin packaging instructions

### Resources
- [Quickstart guide](../docs/quickstart.md) — single-command training from Python
- [Training docs](../docs/training.md) — full trainer API reference
- [Hardware guide](../docs/hardware_guide.md) — tier selection by device
- [All tiers reference](../docs/classifiers.md) — architecture details
- [Search strategies](../docs/search_strategies.md) — tuning the genetic search